# News on TikTok (GESIS) dataset

This notebook analyzes only the GESIS dataset about German-speaking professional news outlets. It checks local availability, structure, paper-count agreement, missingness, reconstructability, and a small number of useful descriptive patterns. All findings are collected at the end.

## Sources and local file

- Paper: [News on TikTok](https://ojs.aaai.org/index.php/ICWSM/article/view/35953)
- Dataset DOI: [10.7802/2819](https://doi.org/10.7802/2819)
- Official CSV: [TikTok-news_data_v2-0-0.csv](https://access.gesis.org/sharing/2863/6673)
- Local folder: `downloaded_datasets/gesis_news/`

A fresh official CSV download was compared byte-for-byte with the local file on 2026-09-17. They were identical.

In [1]:
from pathlib import Path
import ast
import hashlib
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
ROOT = Path.cwd()
if not (ROOT / 'downloaded_datasets').exists() and (ROOT.parent / 'downloaded_datasets').exists():
    ROOT = ROOT.parent
DATA_DIR = ROOT / 'downloaded_datasets' / 'gesis_news'
CSV_PATH = DATA_DIR / 'TikTok-news_data_v2-0-0.csv'
if not CSV_PATH.exists():
    raise FileNotFoundError(CSV_PATH)
print('Local CSV:', CSV_PATH)

Local CSV: /Users/wang/Documents/tiktok/downloaded_datasets/gesis_news/TikTok-news_data_v2-0-0.csv


## 1. Verify local availability and load the data

In [2]:
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

VERIFIED_BYTES = 3_053_641
VERIFIED_SHA256 = '1ea8810709ba2c10aa086a9901e6bb73f788c832a5d3000300519583f53ac220'
local_hash = sha256(CSV_PATH)
display(pd.DataFrame([{
    'local_file': CSV_PATH.name,
    'size_MiB': CSV_PATH.stat().st_size / 1024**2,
    'bytes_match_verified_download': CSV_PATH.stat().st_size == VERIFIED_BYTES,
    'SHA256_matches_verified_download': local_hash == VERIFIED_SHA256,
}]).round({'size_MiB': 2}))

data = pd.read_csv(CSV_PATH, dtype={'M1_video_id': 'string', 'M2_music_id': 'string'})
data['M3_create_time'] = pd.to_datetime(data['M3_create_time'], errors='coerce')
print('Shape:', data.shape)

,local_file,size_MiB,bytes_match_verified_download,SHA256_matches_verified_download
0,TikTok-news_data_v2-0-0.csv,2.91,True,True


Shape: (8623, 37)


## 2. Structure and metadata

Each row represents one TikTok video. The 37 columns combine source identifiers, engagement metadata, hashtags, and human-coded variables.

In [3]:
structure = pd.DataFrame([
    {'content': 'Video ID and TikTok URL', 'available': 'Yes', 'columns': 'M1_video_id, V2_video_link'},
    {'content': 'Music ID and creation time', 'available': 'Yes', 'columns': 'M2_music_id, M3_create_time'},
    {'content': 'Engagement counters', 'available': 'Yes', 'columns': 'M4-M7'},
    {'content': 'Hashtags', 'available': 'Yes', 'columns': 'M8_hashtags'},
    {'content': 'Human annotations', 'available': 'Yes', 'columns': 'V1 and V5-V18'},
    {'content': 'Raw video or audio files', 'available': 'No', 'columns': 'not included'},
    {'content': 'Description, caption, transcript, ASR, OCR', 'available': 'No', 'columns': 'not included'},
    {'content': 'Comment text or comment rows', 'available': 'No', 'columns': 'M6 is only a count'},
])
display(structure)
display(data.head(3))

,content,available,columns
0,Video ID and TikTok URL,Yes,"M1_video_id, V2_video_link"
1,Music ID and creation time,Yes,"M2_music_id, M3_create_time"
2,Engagement counters,Yes,M4-M7
3,Hashtags,Yes,M8_hashtags
4,Human annotations,Yes,V1 and V5-V18
5,Raw video or audio files,No,not included
6,"Description, caption, transcript, ASR, OCR",No,not included
7,Comment text or comment rows,No,M6 is only a count


,V1_coder,V2_video_link,V3_news_outlet,V4_country_of_origin,V5_news_classification,V6_visual_set-up,V7a_images,V7b_text,V7c_screenshots_and_recordings,V7d_graphs_maps_infoanimations,V7e_news_brand_elements,V7f_outlet-related_endcard,V8a_journalists_moderators_voice,V8b_other_peoples_voice,V8c_voice_other_effect,V8d_music,V8e_ambient,V9a_mentions,V9b_call_to_interact_on_tiktok,V9c_call_to_interact_outside_of_tiktok,V10_news_format_type,V11_general_topic,V12_geographic_scope,V13_follow_up,V14_positive_news,V15_negative_news,V16_conflict,V17_scope,V18_temporal_scope,M1_video_id,M2_music_id,M3_create_time,M4_like_count,M5_view_count,M6_comment_count,M7_share_count,M8_hashtags
0,1,https://www.tiktok.com/share/video/72033756326...,tagesanzeiger,CH,1.00,1.00,1.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,2.00,Erklärungen zur Steuererklärung,1.00,0.00,0.00,0.00,0.00,1.00,2.00,7203375632661695749,7203375660541807365,2023-02-23 15:36:28,38,680.00,1,3,"['tagi', 'steuern', 'tagesanzeiger']"
1,0,https://www.tiktok.com/share/video/72029786320...,tagesanzeiger,CH,1.00,1.00,1.00,1.00,1.00,1.00,0.00,0.00,1.00,0.00,0.00,1.00,0.00,0.00,0.00,0.00,3.00,Vanilla Girl Trend,0.00,0.00,0.00,0.00,1.00,1.00,2.00,7202978632086670598,7202978669403998982,2023-02-22 13:55:55,32,653.00,0,2,"['people', 'climatechange', 'vanillagirl', 'zü..."
2,0,https://www.tiktok.com/share/video/72018384221...,tagesanzeiger,CH,1.00,1.00,1.00,1.00,1.00,0.00,1.00,0.00,1.00,0.00,1.00,0.00,0.00,0.00,0.00,1.00,2.00,Hacker-Team Destabilisation von Deomkratien ge...,2.00,0.00,0.00,1.00,0.00,1.00,0.00,7201838422108032262,7086563693013518338,2023-02-19 12:11:22,45,521.00,1,0,"['news', 'tagi', 'teamjorge', 'tagesanzeiger']"


## 3. Check the paper's main counts

In [4]:
count_checks = pd.DataFrame([
    {'measure': 'All videos', 'paper': 8_623, 'downloaded': len(data)},
    {'measure': 'News outlet accounts', 'paper': 18, 'downloaded': data['V3_news_outlet'].nunique()},
    {'measure': 'Videos classified as news', 'paper': 7_978, 'downloaded': int((data['V5_news_classification'] == 1).sum())},
])
count_checks['difference'] = count_checks['downloaded'] - count_checks['paper']
count_checks['identical'] = count_checks['difference'].eq(0)
display(count_checks)

labels = {-99.0: 'unclear', 1.0: 'news', 2.0: 'self-promotion', 3.0: 'branded content', 4.0: 'interactive content', 77.0: 'unavailable'}
classes = data['V5_news_classification'].value_counts().sort_index().rename(index=labels).rename('videos').to_frame()
classes['percent'] = classes['videos'] / len(data) * 100
display(classes.round(2))

,measure,paper,downloaded,difference,identical
0,All videos,8623,8623,0,True
1,News outlet accounts,18,18,0,True
2,Videos classified as news,7978,7978,0,True


,videos,percent
V5_news_classification,,
unclear,111,1.29
news,7978,92.52
self-promotion,172,1.99
branded content,187,2.17
interactive content,106,1.23
unavailable,69,0.80


## 4. Reconstruction and missing values

In [5]:
url_id = data['V2_video_link'].astype('string').str.extract(r'(\d{19})(?:\D*)$', expand=False)
reconstruction = pd.DataFrame([{
    'rows': len(data),
    'unique_video_IDs': data['M1_video_id'].nunique(),
    'duplicate_ID_rows': int(data['M1_video_id'].duplicated().sum()),
    'unique_URLs': data['V2_video_link'].nunique(),
    'URL_ID_exact_matches': int((url_id == data['M1_video_id']).sum()),
    'URL_ID_mismatch_or_missing': int((url_id != data['M1_video_id']).fillna(True).sum()),
}])
display(reconstruction)

missing = pd.DataFrame({
    'missing_count': data.isna().sum(),
    'missing_percent': data.isna().mean() * 100,
}).query('missing_count > 0').sort_values('missing_count', ascending=False)
display(missing.round(2))

,rows,unique_video_IDs,duplicate_ID_rows,unique_URLs,URL_ID_exact_matches,URL_ID_mismatch_or_missing
0,8623,8623,0,8623,8623,0


,missing_count,missing_percent
V7d_graphs_maps_infoanimations,646,7.49
V6_visual_set-up,645,7.48
V7a_images,645,7.48
V18_temporal_scope,645,7.48
V17_scope,645,7.48
V16_conflict,645,7.48
V15_negative_news,645,7.48
V14_positive_news,645,7.48
V13_follow_up,645,7.48
V12_geographic_scope,645,7.48


## 5. Interesting point: reproduce the engagement table

In [6]:
engagement_columns = {'Views': 'M5_view_count', 'Likes': 'M4_like_count', 'Comments': 'M6_comment_count', 'Shares': 'M7_share_count'}
paper = pd.DataFrame({
    'mean': {'Views': 48_878.32, 'Likes': 4_523.64, 'Comments': 200.67, 'Shares': 303.34},
    'std': {'Views': 191_810.48, 'Likes': 17_199.79, 'Comments': 1_655.49, 'Shares': 1_854.64},
    'median': {'Views': 1_645, 'Likes': 707, 'Comments': 22, 'Shares': 3},
    'max': {'Views': 7_105_469, 'Likes': 821_805, 'Comments': 137_075, 'Shares': 75_864},
})
observed = pd.DataFrame({
    label: {'n': data[col].count(), 'mean': data[col].mean(), 'std': data[col].std(), 'min': data[col].min(), 'median': data[col].median(), 'max': data[col].max()}
    for label, col in engagement_columns.items()
}).T
display(observed.round(2))
agreement = observed[['mean', 'std', 'median', 'max']].round(2).eq(paper.round(2))
print('All reported paper values match after rounding to two decimals:', bool(agreement.to_numpy().all()))

,n,mean,std,min,median,max
Views,"8,559.00","48,878.32","191,810.48",0.00,"1,645.00","7,105,469.00"
Likes,"8,623.00","4,523.64","17,199.79",1.00,707.00,"821,805.00"
Comments,"8,623.00",200.67,"1,655.49",0.00,22.00,"137,075.00"
Shares,"8,623.00",303.34,"1,854.64",0.00,3.00,"75,864.00"


All reported paper values match after rounding to two decimals: True


## 6. Interesting point: engagement is strongly concentrated

In [7]:
def top_share(series, fraction=0.01):
    clean = series.dropna().sort_values(ascending=False)
    top_n = max(1, int(np.ceil(len(clean) * fraction)))
    return clean.iloc[:top_n].sum() / clean.sum() * 100

concentration = []
for label, col in engagement_columns.items():
    values = data[col].dropna()
    concentration.append({
        'metric': label, 'mean': values.mean(), 'median': values.median(),
        'mean_to_median_ratio': values.mean() / values.median() if values.median() else np.inf,
        'share_from_top_1_percent': top_share(values),
    })
display(pd.DataFrame(concentration).round(2))

,metric,mean,median,mean_to_median_ratio,share_from_top_1_percent
0,Views,"48,878.32","1,645.00",29.71,28.58
1,Likes,"4,523.64",707.00,6.40,25.20
2,Comments,200.67,22.00,9.12,34.08
3,Shares,303.34,3.00,101.11,44.23


## 7. Interesting point: country and outlet concentration

In [8]:
countries = data['V4_country_of_origin'].value_counts().rename('videos').to_frame()
countries['percent'] = countries['videos'] / len(data) * 100
display(countries.round(2))

outlets = data['V3_news_outlet'].value_counts().rename('videos').to_frame()
outlets['percent'] = outlets['videos'] / len(data) * 100
display(outlets)
print('Top two outlets:', round(outlets.head(2)['percent'].sum(), 2), '% of videos')
print('Top five outlets:', round(outlets.head(5)['percent'].sum(), 2), '% of videos')

,videos,percent
V4_country_of_origin,,
CH,4166,48.31
AT,3143,36.45
DE,1314,15.24


,videos,percent
V3_news_outlet,,
20minuten,1853,21.49
heute.at,1090,12.64
blick,754,8.74
watson_news,537,6.23
derstandard,444,5.15
zeitimbild,424,4.92
bluenews.ch,395,4.58
kleinezeitung,389,4.51
tagesschau,388,4.50


Top two outlets: 34.13 % of videos
Top five outlets: 54.25 % of videos


## 8. Interesting point: hashtag structure and common tags

In [9]:
def parse_hashtags(value):
    if pd.isna(value):
        return [], 'missing'
    text = str(value).strip()
    if text.startswith('['):
        try:
            tags = ast.literal_eval(text)
            return [str(tag).strip().lower() for tag in tags if str(tag).strip()], 'python-list'
        except (ValueError, SyntaxError):
            return [], 'malformed-list'
    return [tag.strip().lower().lstrip('#') for tag in text.split(',') if tag.strip()], 'comma-delimited'

tags = Counter()
formats = Counter()
counts_per_video = []
for value in data['M8_hashtags']:
    parsed, stored_format = parse_hashtags(value)
    tags.update(parsed)
    formats[stored_format] += 1
    counts_per_video.append(len(parsed))

display(pd.DataFrame([{
    'hashtag_occurrences': sum(tags.values()), 'unique_hashtags': len(tags),
    'mean_per_video': np.mean(counts_per_video), 'median_per_video': np.median(counts_per_video),
}]).round(2))
display(pd.DataFrame(tags.most_common(15), columns=['hashtag', 'count']))
display(pd.DataFrame(formats.items(), columns=['stored_format', 'rows']).sort_values('rows', ascending=False))

,hashtag_occurrences,unique_hashtags,mean_per_video,median_per_video
0,52752,11674,6.12,5.00


,hashtag,count
0,news,2035
1,fyp,1723
2,nachrichten,1527
3,20min,1208
4,heutezeitung,943
5,tageszeitungheute,903
6,lernenmittiktok,898
7,fy,867
8,schweiz,815
9,österreich,588


,stored_format,rows
0,python-list,8559
1,comma-delimited,59
2,missing,5


## 9. Availability and collection method

**Availability:** The CSV, codebook, and data report are directly downloadable from GESIS under CC BY 4.0.

**Collection:** Researchers selected major news outlets in Germany, Austria, and Switzerland, located their TikTok channels, and excluded channels that were not mainly producing news. They used the official TikTok Research API in January 2024 to collect metadata for videos published during 2023. They cross-checked the IDs by scrolling channel pages with the Zeeschuimer browser extension and added 64 videos.

**Annotation:** Two trained annotators coded the videos in two phases: September-November 2024 and December 2024-February 2025. Variables cover visual and audio setup, format, topic, geographic scope, news values, and related categories.

**Reconstruction:** The dataset supplies an exact video ID and TikTok URL for every row, but it does not archive the media. Deleted, private, changed, or region-restricted videos may no longer be accessible.

# Findings

1. The official GESIS CSV is genuinely stored locally. A fresh official download was byte-for-byte identical to the local 3,053,641-byte file.
2. The dataset has 8,623 rows and 37 columns. It covers 18 news-outlet accounts and videos published from 1 January to 31 December 2023.
3. It contains metadata and human annotations, but no raw video, audio, descriptions, captions, transcripts, OCR, or comment text.
4. The main paper counts match exactly: 8,623 total videos, 18 outlets, and 7,978 videos classified as news. The published engagement statistics also reproduce exactly after rounding.
5. All 8,623 video IDs and URLs are unique, and every URL ends with the matching video ID. This supports traceability, but not guaranteed access to deleted or private posts.
6. Sixty-four rows lack view counts. The 64 missing values may be related to the 64 videos added during the Zeeschuimer validation, but the CSV alone cannot prove that row-level link.
7. Engagement is highly skewed. The top 1% of videos account for 28.58% of views, 25.20% of likes, 34.08% of comments, and 44.23% of shares. Medians are more representative of a typical post than means.
8. Switzerland contributes 48.31% of rows, Austria 36.45%, and Germany 15.24%. The two largest outlets contribute 34.13% of all videos; the five largest contribute 54.25%.
9. The file contains 52,752 hashtag occurrences and 11,674 unique normalized tags. The most common are `news`, `fyp`, and `nachrichten`.
10. Hashtags use mixed serialization: 8,559 rows use Python-list text, 59 use comma-separated text, and 5 are missing. Analysis code must handle both formats.